# ZMQ Scheduler: Distributed Multi Workers

This notebook explains the multi-GPU service pattern.

The idea is simple. Start one worker per GPU. Put one coordinator in front of the workers. The client connects only to the coordinator. The coordinator chooses an idle worker for each job and forwards events back to the client.

This tutorial demonstrates Goal 1: many GPUs, many independent jobs, higher throughput. It does not split one forecast job across multiple GPUs.

Prerequisites:

- At least two CUDA devices are required for the full demonstration.
- Run `docs/example_era5.ipynb` first, or make sure the ERA5 checkpoint and cache already exist under `ASSET_ROOT`.
- This tutorial uses `download=False`, so it will not contact CDS.
- Always run the cleanup cell at the end.

## Mental Model

There are three process roles.

- Worker 0 owns `cuda:0`.
- Worker 1 owns `cuda:1`.
- The coordinator owns no model. It only routes requests and forwards events.

The client does not talk to workers directly. It talks to the coordinator. This gives users one stable endpoint, even when the service has many GPUs behind it.

If two `era5_pretrained` jobs are submitted at the same time, the coordinator can place one job on each worker.

## What Is ZMQ

ZMQ, also called ZeroMQ, is a lightweight message transport library. It gives Python processes a socket API for sending bytes to each other. In this project, those bytes are JSON-encoded scheduler commands and events.

The scheduler uses two one-way channels at each boundary. The client sends forecast commands through a `PUSH` socket. The coordinator receives them through a `PULL` socket. The coordinator then sends work to workers through another `PUSH` socket, and workers receive work through `PULL` sockets. Events travel back in the opposite direction through the event sockets.

The important point is that the client, coordinator, and workers do not need to live in the same Python process. They can be local subprocesses or separate terminals. If the address starts with `ipc://`, ZMQ uses a local inter-process socket file. If the address starts with `tcp://`, ZMQ uses a TCP port.

This tutorial uses `ipc://` because all processes run on the same machine. A production service often uses `tcp://` so several clients can connect through a stable host and port.

In [ ]:
import os

if os.environ.get("OMP_NUM_THREADS") in (None, "", "0"):
    os.environ["OMP_NUM_THREADS"] = "1"

import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import zmq

from flash_aurora.engine.core.asset_root import normalize_asset_root
from flash_aurora.engine.core.redaction import safe_path
from flash_aurora.scheduler import ForecastClient, ForecastClientConfig, ForecastRequest
from flash_aurora.scheduler.protocol import ForecastEvent
from flash_aurora.scheduler.worker import wait_for_bind

PRESET = "era5_pretrained"
VALID_TIME = "2023-01-01T06:00:00"
TIME_INDEX = 1
ROLLOUT_STEPS = 2
INFERENCE_PRECISION = "bf16_mixed@fp32"
GPU_DEVICES = ("cuda:0", "cuda:1")

ASSET_ROOT: Path | str | None = Path("/root/autodl-tmp/aurora")
ASSET_ROOT = normalize_asset_root(ASSET_ROOT)

SOCKET_DIR = ASSET_ROOT / "scheduler_distributed_workers"
SOCKET_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = ASSET_ROOT / "output" / "scheduler_distributed_workers"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

FRONT_COMMAND_ADDR = f"ipc://{SOCKET_DIR / 'front-commands.ipc'}"
FRONT_EVENT_ADDR = f"ipc://{SOCKET_DIR / 'front-events.ipc'}"

print("asset_root:", safe_path(ASSET_ROOT))
print("front_command_addr:", FRONT_COMMAND_ADDR)
print("front_event_addr:", FRONT_EVENT_ADDR)

In [ ]:
WORKER_JOIN_TIMEOUT_S = 120.0


@dataclass(frozen=True)
class WorkerSpec:
    worker_id: str
    device: str
    command_addr: str
    event_addr: str


def make_worker_specs() -> list[WorkerSpec]:
    specs = []
    for index, device in enumerate(GPU_DEVICES):
        specs.append(
            WorkerSpec(
                worker_id=f"gpu{index}-era5",
                device=device,
                command_addr=f"ipc://{SOCKET_DIR / f'worker-{index}-commands.ipc'}",
                event_addr=f"ipc://{SOCKET_DIR / f'worker-{index}-events.ipc'}",
            )
        )
    return specs


def worker_endpoint_arg(spec: WorkerSpec) -> str:
    return ",".join(
        [
            spec.worker_id,
            PRESET,
            spec.command_addr,
            spec.event_addr,
            spec.device,
            "1",
        ]
    )


def spawn_worker(spec: WorkerSpec) -> subprocess.Popen[bytes]:
    cmd = [
        sys.executable,
        "-m",
        "flash_aurora.scheduler",
        "--preset",
        PRESET,
        "--asset-root",
        str(ASSET_ROOT),
        "--worker-id",
        spec.worker_id,
        "--device",
        spec.device,
        "--capacity",
        "1",
        "--inference-precision",
        INFERENCE_PRECISION,
        "--command-addr",
        spec.command_addr,
        "--event-addr",
        spec.event_addr,
        "--poll-timeout-ms",
        "500",
    ]
    return subprocess.Popen(cmd)


def spawn_coordinator(worker_specs: list[WorkerSpec]) -> subprocess.Popen[bytes]:
    cmd = [
        sys.executable,
        "-m",
        "flash_aurora.scheduler.coordinator_main",
        "--command-addr",
        FRONT_COMMAND_ADDR,
        "--event-addr",
        FRONT_EVENT_ADDR,
    ]
    for spec in worker_specs:
        cmd.extend(["--worker", worker_endpoint_arg(spec)])
    return subprocess.Popen(cmd)


def print_event(event: ForecastEvent) -> None:
    if event.kind == "step" and event.export_path is not None:
        print(f"{event.request_id}: step {event.step}: {safe_path(event.export_path)}")
        return
    if event.kind == "step" and event.valid_time is not None:
        print(f"{event.request_id}: step {event.step} valid_time={event.valid_time}")
        return
    print(f"{event.request_id or 'coordinator'}: {event.kind}")


def terminate_process(proc: subprocess.Popen[bytes] | None) -> None:
    if proc is None or proc.poll() is not None:
        return
    proc.terminate()
    try:
        proc.wait(timeout=WORKER_JOIN_TIMEOUT_S)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait(timeout=10)

## 1. Start One Worker Per GPU

Each worker is an ordinary scheduler worker. The only difference is that each worker receives a different `--device` and different socket addresses.

The coordinator will use these socket addresses later.

In [ ]:
context = zmq.Context.instance()
worker_specs = make_worker_specs()
worker_procs = [spawn_worker(spec) for spec in worker_specs]

for spec in worker_specs:
    wait_for_bind(spec.command_addr)
    print("worker started:", spec.worker_id, spec.device)

## 2. Start The Coordinator

The coordinator is the only endpoint that users need to know.

It receives client commands on `FRONT_COMMAND_ADDR`. It sends client events on `FRONT_EVENT_ADDR`. Internally, it forwards jobs to idle workers.

In [ ]:
coordinator_proc = spawn_coordinator(worker_specs)
wait_for_bind(FRONT_COMMAND_ADDR)
print("coordinator started")

## 3. Connect The Client To The Coordinator

The client uses the same `ForecastClient` as the single-worker tutorial. It does not need to know how many workers exist behind the coordinator.

In [ ]:
client: ForecastClient | None = ForecastClient(
    ForecastClientConfig(
        command_addr=FRONT_COMMAND_ADDR,
        event_addr=FRONT_EVENT_ADDR,
        recv_timeout_ms=600_000,
    ),
    context=context,
)

health = client.health()
print("kind:", health.kind)
print("worker_preset:", health.worker_preset)
print("worker_id:", health.worker_id)
print("worker_capacity:", health.worker_capacity)

if PRESET not in (health.worker_preset or ""):
    raise RuntimeError(f"Coordinator does not report preset {PRESET!r}")

## 4. Submit Two Jobs At Once

This is the multi-worker demonstration.

The client submits two jobs before reading any events. The coordinator should dispatch them to two different idle workers. If both GPUs are available, the two jobs can run at the same time.

The event order is not guaranteed. That is normal. In a parallel service, events from different jobs can interleave.

In [ ]:
requests = [
    ForecastRequest(
        request_id="distributed-job-1",
        preset=PRESET,
        steps=ROLLOUT_STEPS,
        valid_time=VALID_TIME,
        time_index=TIME_INDEX,
        download=False,
        export_dir=str(EXPORT_DIR / "distributed-job-1"),
        sticky_key="user-a",
    ),
    ForecastRequest(
        request_id="distributed-job-2",
        preset=PRESET,
        steps=ROLLOUT_STEPS,
        valid_time=VALID_TIME,
        time_index=TIME_INDEX,
        download=False,
        export_dir=str(EXPORT_DIR / "distributed-job-2"),
        sticky_key="user-b",
    ),
]

assert client is not None
for request in requests:
    client.submit(request)
    print("submitted:", request.request_id)

pending = {request.request_id for request in requests}
while pending:
    event = client.recv_event()
    print_event(event)
    if event.kind == "failed":
        raise RuntimeError(event.error or "forecast failed")
    if event.kind == "completed" and event.request_id in pending:
        pending.remove(event.request_id)

print("all distributed jobs completed")

## 5. Sticky Sessions

`sticky_key` is optional.

If a request has `sticky_key="user-a"`, the coordinator remembers which worker handled that key. Later requests with the same key prefer the same worker when that worker has capacity.

This is useful when a service wants locality for repeated jobs from the same caller. It is not required for correctness.

In [ ]:
sticky_request = ForecastRequest(
    request_id="distributed-sticky-check",
    preset=PRESET,
    steps=1,
    valid_time=VALID_TIME,
    time_index=TIME_INDEX,
    download=False,
    output_mode="metadata_only",
    sticky_key="user-a",
)

assert client is not None
for event in client.forecast(sticky_request):
    print_event(event)

## 6. Cleanup

This cell is not optional.

The client sends `shutdown` to the coordinator. The coordinator forwards shutdown to every worker. The fallback termination logic exists so that a failed notebook cell does not leave GPU-owning subprocesses alive.

In [ ]:
if client is not None:
    try:
        client.shutdown_worker()
    finally:
        client.close()

terminate_process(coordinator_proc)
for proc in worker_procs:
    terminate_process(proc)

time.sleep(0.1)
print("distributed workers scheduler closed")